In [1]:
#Optuna v2 2seed · PY
# Optuna v2: جست‌وجوی ضرایب loss (lambda_eeg, lambda_fnirs, lambda_efgf_reg, lambda_cafe_reg)
# روی معماری نامتقارن تأییدشده (Test=39.93-40.77%، بهترین پیکربندی فعلی).
#
# تفاوت کلیدی نسبت به نسخهٔ قبلی: هر trial با ۲ seed میانگین گرفته می‌شود، چون چک واریانس
# نشان داد std طبیعی بین seed ها (~0.6-0.9 واحد درصد) با اندازهٔ اکثر اثرهایی که دنبالشان
# بودیم قابل‌مقایسه است -- بر پایهٔ Bouthillier et al. (2021), MLSys.
# https://arxiv.org/abs/2103.03098
#
# چیزی که در این کد tune می‌شود: فقط ۴ ضریب تابع loss (Eq. 5-6 مقالهٔ TSFNet).
# معماری (کانال‌ها، dropout=0.6 مشترک) و بقیهٔ هایپرپارامترها (lr, weight_decay, batch_size)
# ثابت نگه داشته شده‌اند -- این‌ها در فازهای بعدی جداگانه tune می‌شوند.

import os
import gc
import math
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import optuna
from optuna.trial import TrialState
import warnings
warnings.filterwarnings('ignore')

# ============================== CONFIGURATION ==============================
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\Preproc run f no beer lambert"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

SPLIT_SEED = 42  # NEW: fixed subject-split seed, independent of model-init seeds used below
random.seed(SPLIT_SEED)
torch.manual_seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# NEW: two model-init seeds averaged per trial (per the variance-check finding)
TRIAL_SEEDS = [42, 123]


# ============================== DATASET (verbatim) ==============================
class MMapHybridDataset(Dataset):
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0

        print(f"Scanning {dataset_name} - {len(subs_list)} subjects...")

        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)

            if not os.path.exists(sub_folder):
                continue

            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"),
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)

            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"),
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)

            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")

            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]

                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}).")
                        continue

                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")

                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue

        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name}!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"opt2_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"opt2_mmap_{dataset_name[:3]}_n.npy")

        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass

        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')

        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()

        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor

    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass


# ============================== MODEL (asymmetric-capacity, verified architecture) ==============================
def pearson_correlation_loss(a, b, eps=1e-8):
    a_flat = a.reshape(-1)
    b_flat = b.reshape(-1)
    a_centered = a_flat - a_flat.mean()
    b_centered = b_flat - b_flat.mean()
    numerator = (a_centered * b_centered).sum()
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps
    pcc = numerator / denominator
    return 1.0 - torch.abs(pcc)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))


class TSFNet_With_fNIRS_2D(nn.Module):
    # Asymmetric-capacity architecture (Test=39.93-40.77% across 3 seeds -- validated best config).
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.6):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        self.eeg_conv1 = nn.Conv3d(1, 12, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.eeg_bn1 = nn.BatchNorm3d(12)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv3d(12, 24, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(24)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_fc = nn.Linear(24 * 4 * 4, 64)

        self.nirs_conv1 = nn.Conv3d(2, 6, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(6)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(6, 12, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(12)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1))
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)
        self.nirs_fc = nn.Linear(12 * 4 * 4, 64)
        self.nirs_channel_align = nn.Conv2d(12, 24, kernel_size=1)

        self.efgf_eeg_attn = nn.Conv2d(24, 1, kernel_size=1)
        self.efgf_nirs_attn = nn.Conv2d(12, 1, kernel_size=1)
        self.alpha = nn.Parameter(torch.tensor(0.0))
        self.gamma = nn.Parameter(torch.tensor(0.0))
        self.fusion_proj = nn.Linear(24 * 4 * 4, 64)

        self.pos_encoder = PositionalEncoding(64, max_len=100)
        self.nirs_seq_proj = nn.Linear(12, 64)
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate2 = nn.Linear(128, 64)

        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)

        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, nirs):
        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_feat_map = x.mean(dim=2)
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))

        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)

        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1)
        nirs_feat_map = y.mean(dim=2)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))

        eeg_attn = torch.sigmoid(self.efgf_eeg_attn(eeg_feat_map))
        nirs_attn = torch.sigmoid(self.efgf_nirs_attn(nirs_feat_map))
        alpha = torch.clamp(self.alpha, 0, 1)
        Phi = alpha * nirs_attn + (1 - alpha) * eeg_attn

        gamma = torch.clamp(self.gamma, 0, 1)
        nirs_feat_map_aligned = self.nirs_channel_align(nirs_feat_map)
        fused_feat_map = gamma * eeg_feat_map + (1 - gamma) * nirs_feat_map_aligned + Phi * nirs_feat_map_aligned
        fused_feat = self.fusion_proj(fused_feat_map.flatten(1))

        nirs_seq = y_spatial.transpose(1, 2)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)
        nirs_proj = self.pos_encoder(nirs_seq_proj)

        fused_seq = fused_feat.unsqueeze(1)
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1)

        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat

        nirs_query = nirs_feat.unsqueeze(1)
        fused_kv = fused_feat.unsqueeze(1)
        enhanced_nirs = self.cafe_attn2(nirs_query, fused_kv, fused_kv).squeeze(1)
        gate3 = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat, enhanced_nirs], dim=-1)))
        final_nirs_feat = gate3 * enhanced_nirs + (1 - gate3) * nirs_feat

        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out = self.fc_fnirs(self.dropout(final_nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))

        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out

        Phi_broadcast = Phi.expand(-1, 24, -1, -1)
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_broadcast, final_nirs_feat, enhanced


# ============================== LOAD DATA ONCE ==============================
print("Loading datasets once (shared across all trials and seeds)...")
train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

criterion = nn.CrossEntropyLoss()


# ============================== SINGLE-SEED TRAIN/EVAL ==============================
def train_one_seed(seed, lambda_eeg, lambda_fnirs, lambda_efgf_reg, lambda_cafe_reg,
                    max_epochs, early_stop_patience, trial=None, seed_index=0):
    torch.manual_seed(seed)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.6).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=5e-2)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    best_val_acc = 0.0
    epochs_no_improve = 0
    ckpt = os.path.join(PROCESSED_DATA_PATH, f'optuna_v2_tmp_seed{seed}.pth')

    for epoch in range(1, max_epochs + 1):
        model.train()
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(DEVICE), nirs_b.to(DEVICE), lbl_b.to(DEVICE)
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)

            L_pred = criterion(pred, lbl_b)
            L_eeg = criterion(eeg_out, lbl_b)
            L_fnirs = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out, lbl_b)
            L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)
            loss = L_class + lambda_efgf_reg * L_efgf + lambda_cafe_reg * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(DEVICE), nirs_b.to(DEVICE), lbl_b.to(DEVICE)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), ckpt)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        # NEW: only the FIRST seed of a trial reports to Optuna for pruning -- a clearly bad trial
        # is rejected after just 1 (cheap) seed, instead of always paying for 2 full seeds.
        if trial is not None and seed_index == 0:
            trial.report(val_acc, epoch)
            if trial.should_prune():
                del model
                gc.collect()
                torch.cuda.empty_cache()
                if os.path.exists(ckpt):
                    os.remove(ckpt)
                raise optuna.TrialPruned()

        if epochs_no_improve >= early_stop_patience:
            break

    del model
    gc.collect()
    torch.cuda.empty_cache()
    if os.path.exists(ckpt):
        os.remove(ckpt)
    return best_val_acc


def train_and_evaluate_2seed(lambda_eeg, lambda_fnirs, lambda_efgf_reg, lambda_cafe_reg,
                              max_epochs, early_stop_patience, trial=None):
    """
    Runs seed[0] first (reports to Optuna for pruning -- bad trials rejected cheaply after 1 seed).
    If seed[0] survives pruning, runs seed[1] too and returns the AVERAGE best_val_acc of both
    seeds, per the variance-check finding that single-seed comparisons are within the noise floor.
    """
    val_acc_0 = train_one_seed(TRIAL_SEEDS[0], lambda_eeg, lambda_fnirs, lambda_efgf_reg, lambda_cafe_reg,
                                max_epochs, early_stop_patience, trial=trial, seed_index=0)
    val_acc_1 = train_one_seed(TRIAL_SEEDS[1], lambda_eeg, lambda_fnirs, lambda_efgf_reg, lambda_cafe_reg,
                                max_epochs, early_stop_patience, trial=trial, seed_index=1)
    avg_val_acc = (val_acc_0 + val_acc_1) / 2.0
    if trial is not None:
        print(f"[trial {trial.number}] seed{TRIAL_SEEDS[0]}={val_acc_0*100:.2f}% "
              f"seed{TRIAL_SEEDS[1]}={val_acc_1*100:.2f}% avg={avg_val_acc*100:.2f}%")
    return avg_val_acc


# ============================== OPTUNA OBJECTIVE (only the 4 loss weights) ==============================
SHORT_MAX_EPOCHS = 15
SHORT_EARLY_STOP_PATIENCE = 10


def objective(trial):
    lambda_eeg = trial.suggest_float("lambda_eeg", 0.05, 1.0, log=True)
    lambda_fnirs = trial.suggest_float("lambda_fnirs", 0.05, 1.0, log=True)
    lambda_efgf_reg = trial.suggest_float("lambda_efgf_reg", 0.1, 2.0, log=True)
    lambda_cafe_reg = trial.suggest_float("lambda_cafe_reg", 0.1, 2.0, log=True)

    avg_val_acc = train_and_evaluate_2seed(
        lambda_eeg, lambda_fnirs, lambda_efgf_reg, lambda_cafe_reg,
        max_epochs=SHORT_MAX_EPOCHS, early_stop_patience=SHORT_EARLY_STOP_PATIENCE, trial=trial
    )
    return avg_val_acc


# ============================== RUN OPTUNA SEARCH ==============================
if __name__ == "__main__":
    N_TRIALS = 8  # NEW: reduced from 15, since each surviving trial now costs ~2x (2 seeds)

    def cooldown_callback(study, trial):
        print(f"[cooldown] استراحت ۳۰ ثانیه‌ای بعد از trial {trial.number}...")
        time.sleep(30)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SPLIT_SEED),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
        storage="sqlite:///optuna_v2_tsfnet_study.db",
        study_name="tsfnet_lambda_search_v2_2seed",
        load_if_exists=True,
    )

    print(f"\nStarting Optuna v2 search: {N_TRIALS} trials, 2 seeds/trial (seed 1 only for surviving trials), TPE, median pruner\n")
    study.optimize(objective, n_trials=N_TRIALS, callbacks=[cooldown_callback])

    completed = [t for t in study.trials if t.state == TrialState.COMPLETE]
    pruned = [t for t in study.trials if t.state == TrialState.PRUNED]

    print("\n" + "=" * 70)
    print(f"Study finished: {len(study.trials)} trials ({len(completed)} completed, {len(pruned)} pruned)")
    print("=" * 70)
    print(f"\nBest trial: #{study.best_trial.number}")
    print(f"  Best avg Val Acc (2 seeds): {study.best_value*100:.2f}%")
    print(f"  Best params: {study.best_trial.params}")

    print("\nAll completed trials:")
    for t in sorted(completed, key=lambda t: t.value, reverse=True):
        print(f"  #{t.number}: avg_val_acc={t.value*100:.2f}% | params={t.params}")

    # ============================== FINAL FULL RUN (2 seeds) WITH BEST PARAMS ==============================
    print("\n" + "=" * 70)
    print("FINAL RUN: retraining with best found loss weights, full epoch budget, 2 seeds, then test eval")
    print("=" * 70)

    best_params = study.best_trial.params
    FULL_MAX_EPOCHS = 100
    FULL_EARLY_STOP_PATIENCE = 15

    final_results = []
    for seed in TRIAL_SEEDS:
        torch.manual_seed(seed)
        train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
        model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.6).to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=5e-2)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

        best_val_acc = 0.0
        epochs_no_improve = 0
        final_ckpt = os.path.join(PROCESSED_DATA_PATH, f'optuna_v2_final_seed{seed}.pth')

        for epoch in range(1, FULL_MAX_EPOCHS + 1):
            model.train()
            for eeg_b, nirs_b, lbl_b in train_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(DEVICE), nirs_b.to(DEVICE), lbl_b.to(DEVICE)
                optimizer.zero_grad()
                pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)
                L_pred = criterion(pred, lbl_b)
                L_eeg = criterion(eeg_out, lbl_b)
                L_fnirs = criterion(fnirs_out, lbl_b)
                L_fusion = criterion(fusion_out, lbl_b)
                L_class = L_pred + best_params["lambda_eeg"] * L_eeg + best_params["lambda_fnirs"] * L_fnirs + L_fusion
                L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
                L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)
                loss = L_class + best_params["lambda_efgf_reg"] * L_efgf + best_params["lambda_cafe_reg"] * L_cafe
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            model.eval()
            val_correct = 0
            with torch.no_grad():
                for eeg_b, nirs_b, lbl_b in val_loader:
                    eeg_b, nirs_b, lbl_b = eeg_b.to(DEVICE), nirs_b.to(DEVICE), lbl_b.to(DEVICE)
                    pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                    val_correct += (pred.argmax(1) == lbl_b).sum().item()
            val_acc = val_correct / len(val_loader.dataset)
            scheduler.step(val_acc)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), final_ckpt)
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
            if epochs_no_improve >= FULL_EARLY_STOP_PATIENCE:
                break

        model.load_state_dict(torch.load(final_ckpt))
        model.eval()
        all_preds_final, all_true = [], []
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in test_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(DEVICE), nirs_b.to(DEVICE), lbl_b.to(DEVICE)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                all_preds_final.extend(pred.argmax(1).cpu().numpy())
                all_true.extend(lbl_b.cpu().numpy())
        test_acc = accuracy_score(all_true, all_preds_final) * 100
        final_results.append({'seed': seed, 'best_val_acc': best_val_acc * 100, 'test_acc': test_acc})
        print(f"[final, seed {seed}] Best Val: {best_val_acc*100:.2f}% | Test: {test_acc:.2f}%")

        del model
        gc.collect()
        torch.cuda.empty_cache()
        if os.path.exists(final_ckpt):
            os.remove(final_ckpt)

    test_accs = [r['test_acc'] for r in final_results]
    print(f"\nFinal test accuracy across {len(TRIAL_SEEDS)} seeds: "
          f"mean={np.mean(test_accs):.2f}%  std={np.std(test_accs):.2f}%  "
          f"values={[f'{v:.2f}' for v in test_accs]}")

    print(f"\nBest loss weights found: lambda_eeg={best_params['lambda_eeg']:.4f}, "
          f"lambda_fnirs={best_params['lambda_fnirs']:.4f}, "
          f"lambda_efgf_reg={best_params['lambda_efgf_reg']:.4f}, "
          f"lambda_cafe_reg={best_params['lambda_cafe_reg']:.4f}")

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed.")

C:\Users\SPACE\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading datasets once (shared across all trials and seeds)...
Scanning Train_Set - 18 subjects...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects...
VP025 verified: 1023 windows
VP005 verified: 1016 windows
VP022 verified: 1026 windows
Streaming clean data to disk memory-map (Validation_Set)...
--> Validation_Set Ready! Total Windows: 3065 | Labels: [1023 1023 1

[I 2026-07-08 11:53:36,279] A new study created in RDB with name: tsfnet_lambda_search_v2_2seed



Starting Optuna v2 search: 8 trials, 2 seeds/trial (seed 1 only for surviving trials), TPE, median pruner



[I 2026-07-08 12:06:47,819] Trial 0 finished with value: 0.5092985318107668 and parameters: {'lambda_eeg': 0.15355286838886856, 'lambda_fnirs': 0.8627358286640178, 'lambda_efgf_reg': 0.8960785365368124, 'lambda_cafe_reg': 0.6009974718380313}. Best is trial 0 with value: 0.5092985318107668.


[trial 0] seed42=52.01% seed123=49.85% avg=50.93%
[cooldown] استراحت ۳۰ ثانیه‌ای بعد از trial 0...


... (10 outputs left out)

[I 2026-07-08 13:26:40,214] Trial 6 finished with value: 0.5058727569331158 and parameters: {'lambda_eeg': 0.19603369861210684, 'lambda_fnirs': 0.5254210669345882, 'lambda_efgf_reg': 0.18187859051288216, 'lambda_cafe_reg': 0.46669637672369235}. Best is trial 0 with value: 0.5092985318107668.


[trial 6] seed42=51.91% seed123=49.27% avg=50.59%
[cooldown] استراحت ۳۰ ثانیه‌ای بعد از trial 6...


[I 2026-07-08 13:37:35,071] Trial 7 finished with value: 0.499184339314845 and parameters: {'lambda_eeg': 0.2949301205216347, 'lambda_fnirs': 0.05746499650110707, 'lambda_efgf_reg': 0.6172115948107072, 'lambda_cafe_reg': 0.1666698328606642}. Best is trial 0 with value: 0.5092985318107668.


[trial 7] seed42=50.57% seed123=49.27% avg=49.92%
[cooldown] استراحت ۳۰ ثانیه‌ای بعد از trial 7...

Study finished: 8 trials (8 completed, 0 pruned)

Best trial: #0
  Best avg Val Acc (2 seeds): 50.93%
  Best params: {'lambda_eeg': 0.15355286838886856, 'lambda_fnirs': 0.8627358286640178, 'lambda_efgf_reg': 0.8960785365368124, 'lambda_cafe_reg': 0.6009974718380313}

All completed trials:
  #0: avg_val_acc=50.93% | params={'lambda_eeg': 0.15355286838886856, 'lambda_fnirs': 0.8627358286640178, 'lambda_efgf_reg': 0.8960785365368124, 'lambda_cafe_reg': 0.6009974718380313}
  #6: avg_val_acc=50.59% | params={'lambda_eeg': 0.19603369861210684, 'lambda_fnirs': 0.5254210669345882, 'lambda_efgf_reg': 0.18187859051288216, 'lambda_cafe_reg': 0.46669637672369235}
  #4: avg_val_acc=50.33% | params={'lambda_eeg': 0.12439367209907215, 'lambda_fnirs': 0.2408207265453542, 'lambda_efgf_reg': 0.364731628491121, 'lambda_cafe_reg': 0.23927528765580638}
  #5: avg_val_acc=49.98% | params={'lambda_eeg': 0.31261